# Getting started

In [1]:
# Use the os package to interact with the environment
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re
import glob
import sys as sys


In [2]:
# show all columns when printing
pd.set_option('display.max_columns', None) 

In [3]:
#set releases
RAW_GENO = '/home/jupyter/workspace/path/to/release7/raw_genotypes'
WGS_vcf = '/home/jupyter/workspace/path/to/release8/wgs/deepvariant_joint_calling/vcfs'
WGS_pfile = '/home/jupyter/workspace/path/to/release8/wgs/deepvariant_joint_calling/plink'

CES_vcf = '/home/jupyter/workspace/path/to/release8/clinical_exomes/deepvariant_joint_calling/vcfs'
CES_pfile = '/home/jupyter/workspace/path/to/release8/clinical_exomes/deepvariant_joint_calling/plink'

# Install Packages

In [4]:
%%capture
%%bash

# Install plink 2.0
cd /home/jupyter/
if test -e /home/jupyter/plink2; then

echo "Plink2 is already installed in /home/jupyter/"
else
echo "Plink2 is not installed"
cd /home/jupyter/

wget https://s3.amazonaws.com/plink2-assets/alpha6/plink2_linux_x86_64_20250122.zip

unzip -o plink2_linux_x86_64_20250122.zip

fi

In [5]:
%%bash

# chmod plink 2 to make sure you have permission to run the program
chmod u+x /home/jupyter/plink2

# Screen the LRRK2 p.L1795F (chr12:40322386:G:T) carriers in GP2 WGS, clinical exome and NBA dataset

In [ ]:
%%bash 

# WGS
WGS_pfile='/home/jupyter/workspace/path/to/release8/wgs/deepvariant_joint_calling/plink'

for pgen in `ls ${WGS_pfile}/*/chr12*.pgen`
do
    ancestry=$(basename "$(dirname "${pgen}")")
    pfile_prefix=$(basename ${pgen} '.pgen')

    /home/jupyter/plink2 --pfile ${WGS_pfile}/${ancestry}/${pfile_prefix} \
                         --chr 12 \
                         --from-bp 40322386  \
                         --to-bp 40322386 \
                         --export A \
                         --out ${ancestry}_WGS
done

In [7]:
# read all *_WGS.raw files

def read_raw(file):
    filename=file.split('.')[0]
    dat = pd.read_csv(file,sep='\t')
    dat['filename']=filename
    return dat
    
wgs=pd.concat(map(read_raw, glob.glob('*_WGS.raw')),axis=0)

# Check the carriers of chr12:40322386:G:T

wgs.loc[(wgs['chr12_40322386_G_T_G']!=2)&(~wgs['chr12_40322386_G_T_G'].isnull())].to_csv('WGS_LRRK2_L1795F_carriers.csv',index=False)

In [ ]:
%%bash

#CES
CES_pfile='/home/jupyter/workspace/path/to/release8/clinical_exomes/deepvariant_joint_calling/plink'

/home/jupyter/plink2 --pfile ${CES_pfile}/chr12 \
                     --chr 12 \
                     --from-bp 40322386  \
                     --to-bp 40322386 \
                     --export A \
                     --out CES


In [9]:
# read CES.raw

ces=pd.read_csv('CES.raw',sep='\t')

# Check the carriers of chr12:40322386:G:T
ces.loc[(ces['chr12_40322386_G_T_G']!=2)&(~ces['chr12_40322386_G_T_G'].isnull())].to_csv('CES_LRRK2_L1795F_carriers.csv')

In [ ]:
%%bash

#NBA

RAW_GENO='/home/jupyter/workspace/path/to/release7/raw_genotypes'

for pgen in `ls ${RAW_GENO}/*/*.pgen`
do
    ancestry=$(basename "$(dirname "${pgen}")")
    pfile_prefix=$(basename ${pgen} '.pgen')

    /home/jupyter/plink2 --pfile ${RAW_GENO}/${ancestry}/${pfile_prefix} \
                         --chr 12 \
                         --from-bp 40322386  \
                         --to-bp 40322386 \
                         --export A \
                         --out ${ancestry}_NBA
done


In [11]:
# read all *_NBA.raw files
    
nba=pd.concat(map(read_raw, glob.glob('*_NBA.raw')),axis=0)

# Check the carriers of chr12:40322386:G:T
nba.loc[(nba['Seq_rs111910483.2_ilmnrev_ilmnF2BT_C']!=2)&(~nba['Seq_rs111910483.2_ilmnrev_ilmnF2BT_C'].isnull())].to_csv('NBA_LRRK2_L1795F_carriers.csv',index=False)
